# 3.6 Birleştirme: Concat ve Append

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/06-concat-and-append.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Combining Datasets: Concat and Append

Verinin en ilgi çekici analizleri çoğu zaman farklı kaynakların birleştirilmesinden çıkar. Bu işlemler iki veri kümesinin basit yan yana eklenmesinden, örtüşmeleri doğru ele alan veritabanı tarzı birleştirme ve join'lere kadar uzanır. Series ve DataFrame bu tür işlemler için tasarlanmıştır; Pandas bu veri düzenleme işlerini hızlı ve doğrudan kılan fonksiyon ve yöntemler içerir.

Burada pd.concat ile Series ve DataFrame birleştirmesine bakacağız; daha sonra Pandas'taki bellek içi pd.merge join'lerine geçeceğiz.

Standart içe aktarmalarla başlayalım:


In [ ]:
# import_pd_np.py
import pandas as pd
import numpy as np



Kolaylık için aşağıdaki örneklerde kullanacağımız belirli biçimde bir DataFrame üreten yardımcı fonksiyonu tanımlayalım:


In [ ]:
# make_df.py
def make_df(cols, ind):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(i) for i in ind]
            for c in cols}
    return pd.DataFrame(data, ind)

# example DataFrame
make_df('ABC', range(3))



Ayrıca birden fazla DataFrame'i yan yana göstermek için kısa bir sınıf tanımlayacağız. Kod, IPython/Jupyter'ın zengin nesne gösterimi için kullandığı özel _repr_html_ yönteminden yararlanır:


In [ ]:
# display_class.py
class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)



Kullanımı aşağıdaki bölümlerde netleşecektir.

## Hatırlatma: NumPy Dizilerinde Birleştirme

Series ve DataFrame birleştirmesi, 2.2 NumPy Dizilerinin Temelleri'nde anlatılan np.concatenate ile benzer davranır. İki veya daha fazla dizinin içeriği tek dizide birleştirilebilir:


In [ ]:
# np_concat_1d.py
x = [1, 2, 3]
y = [4, 5, 6]
z = [7, 8, 9]
np.concatenate([x, y, z])



İlk argüman birleştirilecek dizilerin listesi veya demetidir. Çok boyutlu dizilerde sonucun hangi eksen boyunca birleştirileceğini belirten axis anahtar sözcüğü vardır:


In [ ]:
# np_concat_axis.py
x = [[1, 2],
     [3, 4]]
np.concatenate([x, x], axis=1)



## pd.concat ile Basit Birleştirme

pd.concat, np.concatenate'e benzer sözdizimi sunar; birçok seçenek vardır:


```python
# Pandas v1.3.5 imzası
pd.concat(objs, axis=0, join='outer', ignore_index=False, keys=None,
          levels=None, names=None, verify_integrity=False,
          sort=False, copy=True)
```

*(Jupyter/IPython veya özel ortam gerekir — web sayfasında salt okunur.)*


pd.concat, np.concatenate gibi Series veya DataFrame nesnelerinin basit birleştirmesi için kullanılır:


In [ ]:
# concat_series.py
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([ser1, ser2])



Daha yüksek boyutlu nesneler (DataFrame) için de çalışır:


In [ ]:
# concat_dataframes.py
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
display('df1', 'df2', 'pd.concat([df1, df2])')



Varsayılan davranış satır bazında birleştirmedir (axis=0). np.concatenate gibi birleştirmenin yapılacağı eksen belirtilebilir:


In [ ]:
# concat_axis_columns.py
df3 = make_df('AB', [0, 1])
df4 = make_df('CD', [0, 1])
display('df3', 'df4', "pd.concat([df3, df4], axis='columns')")



axis=1 ile eşdeğerdir; burada daha sezgisel olan axis='columns' kullandık.

### Yinelenen İndeksler

np.concatenate ile pd.concat arasındaki önemli farklardan biri: Pandas birleştirme indeksleri korur — sonuçta yinelenen indeksler olsa bile!


In [ ]:
# concat_duplicate_index.py
x = make_df('AB', [0, 1])
y = make_df('AB', [2, 3])
y.index = x.index  # make indices match
display('x', 'y', 'pd.concat([x, y])')



Sonuçta yinelenen indekslere dikkat edin. DataFrame içinde geçerli olsa da sonuç çoğu zaman istenmez. pd.concat bunu yönetmek için seçenekler sunar.

#### Yinelenen indeksleri hata sayma

Sonuçta örtüşen indeks olmadığını doğrulamak için verify_integrity=True kullanılabilir; yinelenen indeks varsa birleştirme istisna fırlatır:


In [ ]:
# concat_verify_integrity.py
try:
    pd.concat([x, y], verify_integrity=True)
except ValueError as e:
    print("ValueError:", e)



#### İndeksi yoksayma

Bazen indeks önemli değildir; ignore_index=True ile yeni bir tamsayı indeks oluşturulur:


In [ ]:
# concat_ignore_index.py
display('x', 'y', 'pd.concat([x, y], ignore_index=True)')



#### MultiIndex anahtarları ekleme

keys ile kaynaklara etiket verilebilir; sonuç hiyerarşik indeksli bir yapıdır:


In [ ]:
# concat_keys.py
display('x', 'y', "pd.concat([x, y], keys=['x', 'y'])")



3.5 Hiyerarşik İndeksleme'deki araçlarla bu çoklu indeksli DataFrame istediğimiz gösterime dönüştürülebilir.

### Join ile Birleştirme

Kısa örneklerde sütun adları örtüşüyordu. Gerçekte kaynakların sütun kümeleri farklı olabilir; pd.concat bu durumda seçenekler sunar:


In [ ]:
# concat_outer_join.py
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])
display('df5', 'df6', 'pd.concat([df5, df6])')



Veri olmayan girişler varsayılan olarak NA ile doldurulur. join parametresi değiştirilebilir: varsayılan birleşim (join='outer'); join='inner' kesişimdir:


In [ ]:
# concat_inner_join.py
display('df5', 'df6',
        "pd.concat([df5, df6], join='inner')")



Hangi sütunların düşeceğine daha ince kontrol için birleştirmeden önce reindex kullanılabilir:


In [ ]:
# concat_reindex.py
pd.concat([df5, df6.reindex(df5.columns, axis=1)])



### append Yöntemi

Dizi birleştirme çok yaygın olduğundan Series ve DataFrame'de append yöntemi vardır; örneğin pd.concat([df1, df2]) yerine df1.append(df2):


In [ ]:
# df_append.py
display('df1', 'df2', 'df1.append(df2)')



> **Not**
>

Python listelerindeki append ve extend yöntemlerinin aksine Pandas'taki append orijinal nesneyi değiştirmez; birleşik veriyle yeni nesne oluşturur. Yeni indeks ve veri tamponu oluşturduğu için verimli değildir — birden fazla append yerine liste oluşturup tek concat çağrısı genelde daha iyidir.

Sonraki bölümde çoklu kaynaktan veri birleştirmenin daha güçlü yolu olan pd.merge join'lerine bakacağız. concat, append ve ilgili işlevler için Pandas dokümantasyonundaki Merge, join, concat and compare bölümüne bakın.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      İki küçük DataFrame oluşturup pd.concat ile satır ve sütun ekseninde birleştirin:
          
      import pandas as pd
import numpy as np

def make_df(cols, ind):
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

a, b = make_df('AB', [0, 1]), make_df('CD', [0, 1])
print("Satır bazında:\n", pd.concat([a, b]))
print("\nSütun bazında:\n", pd.concat([a, b], axis=1))

> **Not**
>
